In [ ]:
#What is Tokenization in NLP?
#Tokenization is the process of splitting text into smaller units called tokens. These tokens can be:

#Words
#Characters
#Subwords
#Sentences (for sentence tokenization)

#Text: "NLP is amazing!"
#Tokens: ["NLP", "is", "amazing", "!"]

In [ ]:
#| Type                       | Description                               | Example                                                         |
#| -------------------------- | ----------------------------------------- | --------------------------------------------------------------- |
#| **Word Tokenization**      | Splitting text into words                 | "I love NLP" → \["I", "love", "NLP"]                            |
#| **Character Tokenization** | Splitting text into individual characters | "NLP" → \["N", "L", "P"]                                        |
#| **Subword Tokenization**   | Breaks rare words into known subparts     | "unhappiness" → \["un", "happi", "ness"]                        |
#| **Sentence Tokenization**  | Splits text into sentences                | "Hello. I am learning NLP." → \["Hello.", "I am learning NLP."] |


In [ ]:
#Tokenization Techniques / Algorithms
#1. Whitespace / Regex Tokenizer

#Splits text by spaces or punctuation using regular expressions.
#Simple, fast.
#Can break on unwanted characters.
from nltk.tokenize import word_tokenize
word_tokenize("I'm learning NLP!")  # ['I', "'m", 'learning', 'NLP', '!']

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [1]:
# Install dependencies as needed:
!pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter



In [2]:
# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "sid321axn/amazon-alexa-reviews",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

/tmp/ipython-input-2-1901253053.py:5: DeprecationWarning: load_dataset is deprecated and will be removed in a future version.
  df = kagglehub.load_dataset(


ValueError: Unsupported file extension: ''. Supported file extensions are: .csv, .tsv, .json, .jsonl, .xml, .parquet, .feather, .sqlite, .sqlite3, .db, .db3, .s3db, .dl3, .xls, .xlsx, .xlsm, .xlsb, .odf, .ods, .odt

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [ ]:
df = pd.read_csv("your_data.csv")  # Replace with your actual file path

# -------------------------------
# Step 2: Preprocess the data
# -------------------------------
# Convert feedback to binary
df['feedback'] = df['feedback'].apply(lambda x: 1 if x == 'Positive' or x == 1 else 0)

# Fill missing reviews
df['verified_reviews'] = df['verified_reviews'].fillna("")

# Encode 'variation' (optional, not used directly in model here)
le = LabelEncoder()
df['variation'] = le.fit_transform(df['variation'].astype(str))

# -------------------------------
# Step 3: Text Tokenization
# -------------------------------
MAX_WORDS = 5000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['verified_reviews'])

X = tokenizer.texts_to_sequences(df['verified_reviews'])
X = pad_sequences(X, maxlen=MAX_LEN, padding='post')

y = df['feedback'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -------------------------------
# Step 4A: RNN Model
# -------------------------------
def create_rnn_model():
    model = Sequential()
    model.add(Embedding(MAX_WORDS, 64, input_length=MAX_LEN))
    model.add(SimpleRNN(64, return_sequences=False))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# -------------------------------
# Step 4B: LSTM Model
# -------------------------------
def create_lstm_model():
    model = Sequential()
    model.add(Embedding(MAX_WORDS, 64, input_length=MAX_LEN))
    model.add(LSTM(64, return_sequences=False))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# -------------------------------
# Step 5: Train and Evaluate
# -------------------------------
print("Training RNN Model...")
rnn_model = create_rnn_model()
rnn_model.summary()
rnn_model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

print("\nTraining LSTM Model...")
lstm_model = create_lstm_model()
lstm_model.summary()
lstm_model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

# -------------------------------
# Step 6: Evaluation
# -------------------------------
loss_rnn, acc_rnn = rnn_model.evaluate(X_test, y_test, verbose=0)
loss_lstm, acc_lstm = lstm_model.evaluate(X_test, y_test, verbose=0)

print(f"\nRNN Accuracy: {acc_rnn:.4f}")
print(f"LSTM Accuracy: {acc_lstm:.4f}")